# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells” using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) and is accessible via a remote URL.


In [ ]:
# Install mlcroissant
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
metadata = dataset.metadata

# Output basic information about the dataset
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview

Review record sets and fields available in the dataset. All entities are referenced by their `@id` as required by the Croissant specification.

In [ ]:
# List all available record sets and their fields, using @id references

print("Available record sets (with @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")

    # List fields
    if 'field' in rs and rs['field']:
        print("  Fields (@id):")
        for field in rs['field']:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    - {field_id}")
    print("")

Below, as an example, we print out the first record of each record set (using their `@id`).

In [ ]:
# Print the first record of each record set
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"Record Set: {rs_id}")
    try:
        recs = dataset.records(record_set=rs_id)
        first = next(recs)
        print(first)
    except StopIteration:
        print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")
    print("")

## 3. Data Extraction

Here we load all record sets and convert each to a pandas DataFrame for convenient analysis. We always use `@id` to refer to record sets as per the Croissant specification.

Replace the list of `record_set_ids` with the actual `@id`s indicated above. For this demonstration, we will extract the first available record set.


In [ ]:
# Set up record set IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records from Record Set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records) == 0:
        print('  No records found.')
        continue
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head(2))
    print()

# For demonstration, pick the first record set with data
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break
if main_record_set_id is not None:
    print(f"Using first populated record set: {main_record_set_id}\nColumns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set has records.")

## 4. Exploratory Data Analysis (EDA)

Demonstrate basic filtering, normalization, and grouping on one numeric field. For the example below, edit the `numeric_field_id` and `group_field_id` to valid field `@id`s (or column names; often the column name will match the field `@id`) as discovered in sections above.

In [ ]:
# Choose a numeric field and a group field from the extracted dataframe
# Replace these with the correct values for your case (from previous output)
numeric_field_id = None
group_field_id = None
if main_record_set_id is not None:
    cols = dataframes[main_record_set_id].columns.tolist()
    # Attempt to choose a plausible numeric column
    for candidate in cols:
        # Many datasets have 'abundance' or 'MW' (molecular weight) or similar numeric fields
        if 'abun' in candidate.lower() or 'mw' in candidate.lower() or 'count' in candidate.lower() or 'coverage' in candidate.lower():
            numeric_field_id = candidate
            break
    # Try to choose a group field (e.g., 'sample', 'description', 'id')
    for candidate in cols:
        if 'description' in candidate.lower() or 'accession' in candidate.lower() or 'group' in candidate.lower() or 'sample' in candidate.lower():
            group_field_id = candidate
            break

    if numeric_field_id is None or group_field_id is None:
        print(f"Could not auto-select numeric_field/group_field; please set them manually from:")
        print(cols)

    # Proceed if both fields found
    if numeric_field_id is not None:
        df = dataframes[main_record_set_id]
        # Convert to numeric (if necessary, e.g. string columns)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df[numeric_field_id]) # Use the mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean):\n")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize the numeric field
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean by '{group_field_id}':")
            print(grouped.head())
    else:
        print("No numeric field auto-selected for analysis. Please set it manually.")
else:
    print("No data available to analyze.")

## 5. Visualization

We visualize the distribution of the chosen numeric field and group means if available. Plots use matplotlib for compatibility. Adjust as needed for your analysis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None and numeric_field_id in dataframes[main_record_set_id]:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping info is available
    if group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        n_show = min(12, len(group_means))
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_means.index[:n_show], y=group_means.values[:n_show])
        plt.title(f'Mean {numeric_field_id} per {group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No data/field available for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the Croissant-structured FAIR<sup>2</sup> dataset on the mass spectrometry analysis of extracellular vesicles from human mast cells using the `mlcroissant` library. We reviewed the record set and field organization by their `@id`s, extracted records into DataFrames, and performed basic exploratory data analysis and visualization on selected numeric fields. This approach provides a reproducible and rigorous pathway to analyze complex, multi-table open datasets.

You may continue with in-depth data cleaning, domain-specific feature engineering, modeling, or additional visualizations as needed for your bioinformatics or proteomics research workflow.